In [ ]:
# 1. Connecter votre Google Drive (Pour sauvegarder les CSV finaux)
from google.colab import drive
drive.mount('/content/drive')

# 2. Installer le client GDC du TCGA sur Colab
!wget https://gdc.cancer.gov/system/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
!unzip gdc-client_v1.6.1_Ubuntu_x64.zip
!chmod +x gdc-client

# 3. Créer un dossier temporaire pour les images lourdes
!mkdir -p /content/tcga_svs_data

# 4. Télécharger les SVS en utilisant votre fichier manifest (à uploader dans Drive d'abord)
# Remplacez le chemin par celui de votre manifest dans Google Drive
!./gdc-client download -m /content/drive/MyDrive/PFE_DigitalTwin/gdc_manifest.txt -d /content/tcga_svs_data

In [ ]:
import os

manifest_path = '/content/drive/MyDrive/pfe_digital_twin/gdc_manifest.2026-04-15.002227.txt'
download_dir = '/content/tcga_svs_data'

# 1. Verification
if os.path.exists(manifest_path):
    print("✅ Manifest found! Starting the 39 GB download to Colab temporary storage...")

    # Create the destination folder if it doesn't exist
    os.makedirs(download_dir, exist_ok=True)

    # 2. Execute GDC Client
    # -n 8 uses 8 connections to speed up the download
    !./gdc-client download -m {manifest_path} -d {download_dir} -n 8

    print("\n🚀 Download complete! Your SVS files are now in the cloud, safely away from your local hard drive.")
else:
    print("❌ Manifest STILL NOT FOUND. Please check that you renamed it exactly to 'gdc_manifest.txt' and placed it in 'MyDrive/PFE_DigitalTwin/'")

In [ ]:
print('Listing files in /content/drive/MyDrive/pfe_digital_twin/ to confirm manifest presence:')
!ls -F /content/drive/MyDrive/pfe_digital_twin/

In [ ]:
print('Listing files in /content/drive/MyDrive/ to help locate the manifest:')
!ls -F /content/drive/MyDrive/

In [ ]:
import os

download_dir = '/content/tcga_svs_data'
print(f"Listing .svs files in {download_dir}:")
!find {download_dir} -type f -name "*.svs"

In [ ]:
import os

manifest_path = '/content/drive/MyDrive/pfe_digital_twin/gdc_manifest.2026-04-15.002227.txt'
download_dir = '/content/tcga_svs_data'

# 1. Compter les fichiers attendus dans le Manifest (moins la ligne d'en-tête)
with open(manifest_path, 'r') as f:
    expected_files = sum(1 for line in f) - 1

# 2. Compter les fichiers SVS réellement téléchargés
downloaded_svs = sum(1 for root, dirs, files in os.walk(download_dir) for file in files if file.endswith('.svs'))

print("📊 Rapport de Diagnostic du Digital Twin :")
print(f"Fichiers attendus (Manifest) : {expected_files}")
print(f"Fichiers téléchargés sur Colab : {downloaded_svs}")

# 3. Logique de résolution
if expected_files == downloaded_svs:
    print("\n⚠️ Le téléchargement est complet par rapport au manifest. Si vous attendiez 39 Go, c'est que votre fichier n'est peut-être pas complet. Vous devez retourner sur le site du GDC et regénérer un manifest complet.")
elif downloaded_svs < expected_files:
    print("\n✅ Le manifest est bon, mais le téléchargement a été interrompu.")
    print("Exécutez de nouveau la commande gdc-client. Il reprendra automatiquement là où il s'est arrêté sans tout recommencer.")

In [ ]:
manifest_path = '/content/drive/MyDrive/pfe_digital_twin/gdc_manifest.2026-04-15.002227.txt'
download_dir = '/content/tcga_svs_data'

print("Reprise du téléchargement pour le(s) fichier(s) manquant(s)...")
!./gdc-client download -m {manifest_path} -d {download_dir} -n 8
print("\n✅ Vérification et téléchargement terminés !")

In [ ]:
!wc -l /content/drive/MyDrive/pfe_digital_twin/gdc_manifest.2026-04-15.002227.txt

In [ ]:
import os

# Pointing exactly to your new file
manifest_path_txt = '/content/drive/MyDrive/PFE_DigitalTwin/gdc_manifest_full.2026-04-27.232209.txt'
manifest_path_no_ext = '/content/drive/MyDrive/PFE_DigitalTwin/gdc_manifest_full.2026-04-27.232209'

# Auto-detect which one exists
manifest_path = manifest_path_txt if os.path.exists(manifest_path_txt) else manifest_path_no_ext
download_dir = '/content/tcga_svs_data'

if os.path.exists(manifest_path):
    # Verify line count to be absolutely sure before we pull 39GB
    with open(manifest_path, 'r') as f:
        lines = sum(1 for line in f) - 1

    print(f"📊 Verified: {lines} files detected in the manifest.")
    print("🚀 Initiating 39.38 GB transfer to Colab Temporary Storage. Please wait...")

    # Run gdc-client with 10 concurrent connections to maximize speed
    !./gdc-client download -m {manifest_path} -d {download_dir} -n 10

    print("\n✅ Transfer Complete. All SVS files are now in the cloud.")
else:
    print("❌ Manifest not found in Drive. Did you upload it to the 'PFE_DigitalTwin' folder?")

In [ ]:
import os

# Pointing exactly to the newly found file path
manifest_path = '/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin/gdc_manifest_full.2026-04-27.232209.txt'
download_dir = '/content/tcga_svs_data'

if os.path.exists(manifest_path):
    # Verify line count to be absolutely sure before we pull 39GB
    with open(manifest_path, 'r') as f:
        lines = sum(1 for line in f) - 1

    print(f"📊 Verified: {lines} files detected in the manifest.")
    print("🚀 Initiating 39.38 GB transfer to Colab Temporary Storage. Please wait...")

    # Run gdc-client with 10 concurrent connections to maximize speed
    !./gdc-client download -m {manifest_path} -d {download_dir} -n 10

    print("\n✅ Transfer Complete. All SVS files are now in the cloud.")
else:
    print("❌ Manifest not found. Please verify your Google Drive connection.")

In [ ]:
!find /content/drive/ -name "gdc_manifest_full.2026-04-27.232209*"

In [ ]:
import os
import subprocess

# 0. Installer QuPath si nécessaire
qupath_bin = '/content/QuPath/bin/QuPath'
if not os.path.exists(qupath_bin):
    print("⏳ Téléchargement et installation de QuPath...")
    subprocess.run(['wget', 'https://github.com/qupath/qupath/releases/download/v0.4.3/QuPath-0.4.3-Linux.tar.xz', '-q'])
    subprocess.run(['tar', '-xf', 'QuPath-0.4.3-Linux.tar.xz'])
    subprocess.run(['mv', 'QuPath-0.4.3-Linux', '/content/QuPath'])
    print("✅ QuPath installé !")

# Donner les droits d'exécution
subprocess.run(['chmod', '+x', qupath_bin])

# 1. Configuration des dossiers
svs_dir = '/content/tcga_svs_data'
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports'
os.makedirs(export_dir, exist_ok=True)

# 2. Création du script Groovy (Instructions pour QuPath)
groovy_script_path = '/content/cell_detection.groovy'
groovy_code = """
// Définir le type d'image (H&E)
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// Exécuter la Positive Cell Detection (Paramètres standards NSCLC)
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD",  "requestedPixelSizeMicrons": 0.5,  "backgroundRadiusMicrons": 8.0,  "medianRadiusMicrons": 0.0,  "sigmaMicrons": 1.5,  "minAreaMicrons": 10.0,  "maxAreaMicrons": 400.0,  "threshold": 0.1,  "cellExpansionMicrons": 5.0,  "includeNuclei": true,  "smoothBoundaries": true,  "makeMeasurements": true}');

// Exporter UNIQUEMENT les mesures (CSV)
def imageData = getCurrentImageData()
def name = GeneralTools.getNameWithoutExtension(imageData.getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports', name + '_cells.csv')
saveDetectionMeasurements(path)
print("✅ Export réussi pour : " + name)
"""

with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# 3. Lancer le traitement par lots (Batch Processing)
print(f"🚀 Début de l'analyse QuPath sur {len(os.listdir(svs_dir))} dossiers...")

# Parcourir chaque dossier téléchargé pour trouver les SVS
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_path = os.path.join(root, file)
            print(f"\n🔬 Traitement en cours : {file}...")

            # Commande console pour exécuter QuPath Headless
            command = [
                '/content/QuPath/bin/QuPath',
                'script', groovy_script_path,
                '-i', svs_path
            ]

            # Exécution (On masque les logs verbeux de QuPath pour garder la console propre)
            subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

print("\n🎉 TERMINÉ ! Tous les CSV sont maintenant dans votre Google Drive.")

In [ ]:
import os
import subprocess

svs_dir = '/content/tcga_svs_data'
export_dir_test = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Test'
os.makedirs(export_dir_test, exist_ok=True)

# Trouver une image pour le test
svs_test_file = None
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_test_file = os.path.join(root, file)
            break
    if svs_test_file:
        break

# Script Groovy avec paramètres permissifs
groovy_script_path = '/content/cell_detection_permissive.groovy'
groovy_code = """
// Définir le type d'image (H&E)
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// Paramètres permissifs : threshold=0.05, minArea=5.0, maxArea=1000.0
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD",  "requestedPixelSizeMicrons": 0.5,  "backgroundRadiusMicrons": 8.0,  "medianRadiusMicrons": 0.0,  "sigmaMicrons": 1.5,  "minAreaMicrons": 5.0,  "maxAreaMicrons": 1000.0,  "threshold": 0.05,  "cellExpansionMicrons": 5.0,  "includeNuclei": true,  "smoothBoundaries": true,  "makeMeasurements": true}');

def imageData = getCurrentImageData()
def name = GeneralTools.getNameWithoutExtension(imageData.getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Test', name + '_permissive_cells.csv')
saveDetectionMeasurements(path)
print("✅ Export test réussi pour : " + name)
"""

with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

if svs_test_file:
    print(f"🚀 Test avec paramètres permissifs sur : {os.path.basename(svs_test_file)}")

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_script_path,
        '-i', svs_test_file
    ]

    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    # Vérifier le résultat
    test_csv_name = os.path.basename(svs_test_file).replace('.svs', '_permissive_cells.csv')
    test_csv_path = os.path.join(export_dir_test, test_csv_name)

    if os.path.exists(test_csv_path):
        with open(test_csv_path, 'r') as f:
            lines = sum(1 for _ in f)
            cells_detected = lines - 1 if lines > 0 else 0
        print(f"\n✅ Test terminé ! {cells_detected:,} cellules ont été détectées avec ces paramètres.")
    else:
        print("\n❌ Le fichier CSV n'a pas été généré, il y a eu une erreur.")
else:
    print("Aucun fichier SVS trouvé.")

In [ ]:
import os
import subprocess

svs_dir = '/content/tcga_svs_data'

# Trouver le premier fichier SVS pour le test
svs_test_file = None
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_test_file = os.path.join(root, file)
            break
    if svs_test_file:
        break

if svs_test_file:
    print(f"🔬 Test de lecture QuPath sur : {os.path.basename(svs_test_file)}\n")

    # Script Groovy de diagnostic
    groovy_diag_script = '/content/check_image.groovy'
    groovy_code = """
    def imageData = getCurrentImageData()
    if (imageData == null) {
        print("❌ ERREUR: Impossible de lire l'image.")
        return
    }
    def server = imageData.getServer()
    print("✅ SUCCÈS: Image chargée correctement !")
    print("📊 Type de lecteur : " + server.getClass().getSimpleName())
    print("📏 Largeur : " + server.getWidth() + " pixels")
    print("📏 Hauteur : " + server.getHeight() + " pixels")
    """

    with open(groovy_diag_script, 'w') as f:
        f.write(groovy_code)

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_diag_script,
        '-i', svs_test_file
    ]

    # Exécuter et capturer la sortie
    result = subprocess.run(command, capture_output=True, text=True)

    print("--- Résultat QuPath ---")
    # On filtre la sortie pour ne garder que nos messages de diagnostic
    for line in result.stdout.split('\n'):
        if "✅" in line or "❌" in line or "📊" in line or "📏" in line:
            print(line.strip())
    print("-----------------------")

    if result.stderr:
        print("\nErreurs signalées par QuPath :\n", result.stderr[:500])
else:
    print("Aucun fichier SVS trouvé pour le test.")

In [ ]:
import os
import subprocess

svs_dir = '/content/tcga_svs_data'

# Trouver un fichier SVS pour le test
svs_test_file = None
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_test_file = os.path.join(root, file)
            break
    if svs_test_file:
        break

if svs_test_file:
    print(f"🔬 Analyse des vecteurs de coloration sur : {os.path.basename(svs_test_file)}\n")

    # Script Groovy pour extraire les vecteurs H&E
    groovy_stain_script = '/content/check_stains.groovy'
    groovy_code = """
    // Forcer le type H&E pour initialiser les stains par défaut de QuPath
    setImageType('BRIGHTFIELD_H_E');

    def imageData = getCurrentImageData()
    def stains = imageData.getColorDeconvolutionStains()

    print("--- VECTEURS DE COLORATION (STAINS) ---")
    if (stains != null) {
        print("Nom : " + stains.getName())

        def s1 = stains.getStain(1)
        print("🟩 Stain 1: " + s1.toString())

        def s2 = stains.getStain(2)
        print("🟪 Stain 2: " + s2.toString())

        def s3 = stains.getStain(3)
        print("⬛ Stain 3: " + s3.toString())

        // print("Background : " + stains.getMaxBackground()) // Removed due to error
    } else {
        print("❌ Aucun vecteur de coloration trouvé.")
    }
    print("---------------------------------------")
    """

    with open(groovy_stain_script, 'w') as f:
        f.write(groovy_code)

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_stain_script,
        '-i', svs_test_file
    ]

    # Exécuter et capturer la sortie
    result = subprocess.run(command, capture_output=True, text=True)

    # Filtrer l'output pour afficher uniquement nos résultats
    for line in result.stdout.split('\n'):
        if "---" in line or "🟩" in line or "🟪" in line or "⬛" in line or "Nom :" in line or "Background :" in line or "❌" in line:
            print(line.strip())

else:
    print("Aucun fichier SVS trouvé.")

In [ ]:
import os
import subprocess

svs_dir = '/content/tcga_svs_data'
export_dir_test = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Test_Tissue'
os.makedirs(export_dir_test, exist_ok=True)

# Trouver un fichier SVS pour le test
svs_test_file = None
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_test_file = os.path.join(root, file)
            break
    if svs_test_file:
        break

# Script Groovy combinant Détection de Tissu + Détection de Cellules
groovy_script_path = '/content/tissue_and_cell_detection.groovy'
groovy_code = """
// 1. Définir le type d'image (H&E)
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// 2. Nettoyer les anciennes annotations éventuelles
clearAnnotations();

// 3. Détection du tissu (Crée une annotation autour du tissu pour ignorer le fond blanc)
runPlugin('qupath.imagej.detect.tissue.SimpleTissueDetection2', '{"threshold": 215,  "requestedPixelSizeMicrons": 20.0,  "minAreaMicrons": 10000.0,  "maxHoleAreaMicrons": 1000.0,  "darkIsPositive": false,  "smoothRadius": 2,  "makeAnnotations": true,  "colorByRGB": false}');

// 4. Sélectionner l'annotation de tissu nouvellement créée
selectAnnotations();

// 5. Exécuter la détection de cellules UNIQUEMENT dans le tissu sélectionné
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD",  "requestedPixelSizeMicrons": 0.5,  "backgroundRadiusMicrons": 8.0,  "medianRadiusMicrons": 0.0,  "sigmaMicrons": 1.5,  "minAreaMicrons": 10.0,  "maxAreaMicrons": 400.0,  "threshold": 0.1,  "cellExpansionMicrons": 5.0,  "includeNuclei": true,  "smoothBoundaries": true,  "makeMeasurements": true}');

// 6. Exporter les mesures
def imageData = getCurrentImageData()
def name = GeneralTools.getNameWithoutExtension(imageData.getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Test_Tissue', name + '_cells_with_tissue.csv')
saveDetectionMeasurements(path)
print("✅ Export réussi avec détection de tissu pour : " + name)
"""

with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

if svs_test_file:
    print(f"🚀 Test de détection Tissu + Cellules sur : {os.path.basename(svs_test_file)}\nCela peut prendre 1 à 2 minutes...")

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_script_path,
        '-i', svs_test_file
    ]

    # Exécution du script
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    # Vérifier le résultat du CSV
    test_csv_name = os.path.basename(svs_test_file).replace('.svs', '_cells_with_tissue.csv')
    test_csv_path = os.path.join(export_dir_test, test_csv_name)

    if os.path.exists(test_csv_path):
        with open(test_csv_path, 'r') as f:
            lines = sum(1 for _ in f)
            cells_detected = lines - 1 if lines > 0 else 0
        print(f"\n✅ Test terminé ! {cells_detected:,} cellules ont été détectées.")
    else:
        print("\n❌ Le fichier CSV n'a pas été généré.")
else:
    print("Aucun fichier SVS trouvé.")

In [ ]:
import os
import glob

export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports'
csv_files = glob.glob(os.path.join(export_dir, '*.csv'))

total_cells = 0
file_count = 0

print(f"Comptage des cellules dans {len(csv_files)} fichiers CSV...")

for file in csv_files:
    try:
        # On compte les lignes du fichier de manière efficace (sans tout charger en RAM)
        with open(file, 'r') as f:
            # Compter toutes les lignes
            lines = sum(1 for _ in f)
            if lines > 1:
                # Ajouter au total (lignes totales - 1 pour l'en-tête)
                total_cells += (lines - 1)
        file_count += 1
    except Exception as e:
        print(f"Erreur lors de la lecture de {os.path.basename(file)} : {e}")

print(f"\n✅ Analyse terminée !")
print(f"📊 Nombre total de fichiers lus : {file_count}")
print(f"🦠 Nombre total de cellules détectées : {total_cells:,}")

In [ ]:
import tarfile
import pandas as pd
import os
import glob

# 1. Configuration des chemins
# Recherche automatique du fichier tar dans le dossier PFE ou ses variantes
search_paths = [
    '/content/drive/MyDrive/PFE_DigitalTwin/clinical.cart*.tar*',
    '/content/drive/MyDrive/pfe_digital_twin/clinical.cart*.tar*',
    '/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin/clinical.cart*.tar*'
]

tar_path = None
for path in search_paths:
    matches = glob.glob(path)
    if matches:
        tar_path = matches[0]
        break

extract_dir = '/content/tcga_clinical_data'
os.makedirs(extract_dir, exist_ok=True)

if tar_path is None:
    print("❌ Erreur : Impossible de trouver un fichier 'clinical.cart...tar' dans votre Drive.")
    print("Veuillez vérifier que le fichier a bien été uploadé et que le nom correspond.")
else:
    # 2. Extraction sécurisée de l'archive tar
    print(f"📦 Archive clinique trouvée : {tar_path}")
    print("📦 Extraction en cours...")

    def extract_tar(tar, path):
        if hasattr(tarfile, 'data_filter'):
            tar.extractall(path=path, filter='data')
        else:
            tar.extractall(path=path)

    try:
        with tarfile.open(tar_path, 'r') as tar:
            extract_tar(tar, extract_dir)
    except Exception as e:
        # Parfois l'extension est .tar mais le fichier est compressé en gzip (.tar.gz)
        try:
            with tarfile.open(tar_path, 'r:gz') as tar:
                extract_tar(tar, extract_dir)
        except Exception as e2:
            print(f"❌ Erreur lors de l'extraction : {e2}")

    # 3. Localisation et lecture du fichier TSV
    clinical_tsv_path = None
    for root, dirs, files in os.walk(extract_dir):
        if 'clinical.tsv' in files:
            clinical_tsv_path = os.path.join(root, 'clinical.tsv')
            break

    if clinical_tsv_path:
        # Lecture avec Pandas
        df_clinical = pd.read_csv(clinical_tsv_path, sep='\t')
        print(f"✅ Succès : {df_clinical.shape[0]} entrées cliniques chargées.")

        # Isoler les colonnes essentielles pour identifier le patient (Case ID)
        colonnes_cles = ['case_submitter_id', 'project_id', 'primary_diagnosis', 'vital_status']
        colonnes_presentes = []
        for col in colonnes_cles:
            # Chercher la colonne même s'il y a un préfixe (ex: cases.case_submitter_id)
            matching_cols = [c for c in df_clinical.columns if col in c]
            if matching_cols:
                colonnes_presentes.append(matching_cols[0])

        df_clinical_clean = df_clinical[colonnes_presentes].drop_duplicates()
        print("\n📊 Aperçu de la base clinique (Prête pour la Master Table) :")
        print(df_clinical_clean.head())

        # Sauvegarde en CSV léger pour Unity / Master Table
        export_path = '/content/drive/MyDrive/PFE_DigitalTwin/Clinical_Cleaned.csv'
        os.makedirs(os.path.dirname(export_path), exist_ok=True)
        df_clinical_clean.to_csv(export_path, index=False)
        print(f"\n💾 Fichier clinique nettoyé sauvegardé : {export_path}")
    else:
        print("❌ Erreur : Fichier 'clinical.tsv' introuvable dans l'archive.")

In [ ]:
import pandas as pd
import os
import glob

# 1. Recherche automatique du fichier dans le Drive
search_paths = [
    '/content/drive/MyDrive/PFE_DigitalTwin/Master_Twin_Data_Final.csv',
    '/content/drive/MyDrive/pfe_digital_twin/Master_Twin_Data_Final.csv',
    '/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin/Master_Twin_Data_Final.csv'
]

final_data_path = None
for path in search_paths:
    matches = glob.glob(path)
    if matches:
        final_data_path = matches[0]
        break

if final_data_path is None:
    print("❌ Erreur : Impossible de trouver le fichier 'Master_Twin_Data_Final.csv' dans votre Drive.")
else:
    # 2. Chargement du fichier
    print(f"📊 Fichier trouvé : {final_data_path}")
    print("📊 Chargement de la Master Twin Data Final...")
    df_twin = pd.read_csv(final_data_path)

    # 3. Sélection stricte des données pour le Digital Twin 3D (Unity)
    # Nous ne gardons que l'ID, la survie et les 5 mutations majeures pour le rendu des couleurs
    colonnes_unity = [
        'Tumor ID', 'Survival', 'TP53', 'EGFR', 'KRAS', 'STK11', 'KEAP1'
    ]

    # Filtrage intelligent (on ne garde que les colonnes qui existent vraiment)
    colonnes_presentes = [col for col in colonnes_unity if col in df_twin.columns]
    df_unity_export = df_twin[colonnes_presentes].copy()

    # 4. Standardisation pour le TCGA
    df_unity_export = df_unity_export.rename(columns={'Tumor ID': 'TCGA_Barcode'})

    print("\n✅ Aperçu du fichier allégé prêt pour le croisement spatial :")
    print(df_unity_export.head())

    # 5. Export vers le Drive
    export_path = '/content/drive/MyDrive/PFE_DigitalTwin/Master_Patient_Profiles.csv'
    os.makedirs(os.path.dirname(export_path), exist_ok=True)
    df_unity_export.to_csv(export_path, index=False)
    print(f"\n💾 Fichier Profils Patients sauvegardé : {export_path}")


In [ ]:
import os

# Liste des chemins possibles pour votre dossier de projet
folders_to_check = [
    '/content/drive/MyDrive/PFE_DigitalTwin',
    '/content/drive/MyDrive/pfe_digital_twin',
    '/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin'
]

print("🔍 Vérification de l'accès aux dossiers du projet :\n")

folder_found = False
for folder in folders_to_check:
    if os.path.exists(folder):
        print(f"✅ Dossier TROUVÉ : {folder}")
        print("\nContenu du dossier :")
        # Liste les fichiers dans ce dossier
        for f in os.listdir(folder):
            print(f"  - {f}")
        folder_found = True
        print("-" * 40)
    else:
        print(f"❌ Dossier INTROUVABLE : {folder}")

if not folder_found:
    print("\n⚠️ Aucun dossier de projet n'a été trouvé. Vérifiez que votre Drive est bien monté et que le dossier existe.")


In [ ]:
import os
import subprocess

svs_dir = '/content/tcga_svs_data'
export_dir_unity = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
os.makedirs(export_dir_unity, exist_ok=True)

# 1. Création du script Groovy pour QuPath
groovy_script_path = '/content/batch_tissue_cell_detection.groovy'
groovy_code = """
// Initialisation
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');
clearAnnotations();

// Détection du tissu (Zone tumorale)
runPlugin('qupath.imagej.detect.tissue.SimpleTissueDetection2', '{"threshold": 215,  "requestedPixelSizeMicrons": 20.0,  "minAreaMicrons": 10000.0,  "maxHoleAreaMicrons": 1000.0,  "darkIsPositive": false,  "smoothRadius": 2,  "makeAnnotations": true,  "colorByRGB": false}');

// Assigner la classe "Tumor" aux annotations détectées pour que Unity puisse les identifier
def tissueAnnotations = getAnnotationObjects()
tissueAnnotations.forEach { it.setPathClass(getPathClass("Tumor")) }
fireHierarchyUpdate()

// Détection des Cellules (Positives / Négatives)
selectAnnotations();
// Le paramètre threshold (0.1) détermine la positivité de la cellule
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD",  "requestedPixelSizeMicrons": 0.5,  "backgroundRadiusMicrons": 8.0,  "medianRadiusMicrons": 0.0,  "sigmaMicrons": 1.5,  "minAreaMicrons": 10.0,  "maxAreaMicrons": 400.0,  "threshold": 0.1,  "cellExpansionMicrons": 5.0,  "includeNuclei": true,  "smoothBoundaries": true,  "makeMeasurements": true}');

// Export CSV pour Unity
def imageData = getCurrentImageData()
def name = GeneralTools.getNameWithoutExtension(imageData.getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
print("✅ Traitement et export réussis pour : " + name)
"""

with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# 2. Récupérer toutes les images SVS
svs_files = []
for root, dirs, files in os.walk(svs_dir):
    for file in files:
        if file.endswith('.svs'):
            svs_files.append(os.path.join(root, file))

total_files = len(svs_files)
print(f"🚀 Lancement du traitement par lots sur {total_files} images SVS...")

# 3. Traitement par lots
for i, svs_path in enumerate(svs_files, 1):
    file_name = os.path.basename(svs_path)
    print(f"[{i}/{total_files}] 🔬 Analyse en cours : {file_name}...")

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_script_path,
        '-i', svs_path
    ]

    # Exécution (on masque les logs verbeux de QuPath pour garder la console propre)
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

print("\n🎉 BATCH TERMINÉ ! Tous les CSV sont exportés et prêts pour Unity.")


🚀 Lancement du traitement par lots sur 129 images SVS...
[1/129] 🔬 Analyse en cours : TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b34-5b168c2496c7.svs...
[2/129] 🔬 Analyse en cours : TCGA-44-6147-01B-05-BS5.B838E2DC-8869-4C72-9F1D-A066FF307579.svs...
[3/129] 🔬 Analyse en cours : TCGA-38-6178-01A-01-BS1.45e6381d-637a-4b27-aae9-1bbee2c14944.svs...
[4/129] 🔬 Analyse en cours : TCGA-64-1681-01A-01-TS1.95ef9027-459a-4796-97a2-773e7cfed3ee.svs...
[5/129] 🔬 Analyse en cours : TCGA-69-7765-01Z-00-DX1.ac389366-febb-488c-9190-fe00bc07cd20.svs...
[6/129] 🔬 Analyse en cours : TCGA-44-6147-01B-03-BS3.B5B5233D-4157-4A68-B260-A264D54E98E1.svs...
[7/129] 🔬 Analyse en cours : TCGA-55-6980-11A-01-TS1.7d9af225-8aea-4609-9e58-fb190f63f13f.svs...
[8/129] 🔬 Analyse en cours : TCGA-50-6595-11A-01-TS1.9aa7c8df-9306-4f4c-8f85-a70d4cbdbcbf.svs...
[9/129] 🔬 Analyse en cours : TCGA-49-4494-01Z-00-DX6.3791a027-7ddb-429c-99a5-bb84a4307550.svs...
[10/129] 🔬 Analyse en cours : TCGA-91-6847-11A-01-TS1.bed7fd2b-0ae1-49

KeyboardInterrupt: 

In [ ]:
import os
import subprocess
import time
from google.colab import drive

print("🔌 PHASE 1 : Reconnexion au stockage permanent...")
drive.mount('/content/drive')

print("⚙️ PHASE 2 : Restauration de l'infrastructure (QuPath & GDC Client)...")
# Installation silencieuse pour ne pas polluer la console
!wget -q https://gdc.cancer.gov/system/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
!unzip -q -o gdc-client_v1.6.1_Ubuntu_x64.zip
!chmod +x gdc-client
!wget -q https://github.com/qupath/qupath/releases/download/v0.4.3/QuPath-0.4.3-Linux.tar.xz
!tar -xf QuPath-0.4.3-Linux.tar.xz

print("📥 PHASE 3 : Rapatriement des données SVS (Prenez un café, ~15 min)...")
manifest_path = '/content/drive/MyDrive/PFE_DigitalTwin/gdc_manifest_full.2026-04-27.232209.txt'
svs_dir = '/content/tcga_svs_data'
os.makedirs(svs_dir, exist_ok=True)
!./gdc-client download -m {manifest_path} -d {svs_dir} -n 10

print("\n🚀 PHASE 4 : Reprise de l'extraction spatiale QuPath...")
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
os.makedirs(export_dir, exist_ok=True)

# Recréation du script Groovy
groovy_script_path = '/content/batch_detection.groovy'
groovy_code = """
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');
createSelectAllObject(true);
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD", "requestedPixelSizeMicrons": 0.5, "backgroundRadiusMicrons": 8.0, "medianRadiusMicrons": 0.0, "sigmaMicrons": 1.5, "minAreaMicrons": 5.0, "maxAreaMicrons": 1000.0, "threshold": 0.05, "cellExpansionMicrons": 5.0, "includeNuclei": true, "smoothBoundaries": true, "makeMeasurements": true}');
def name = GeneralTools.getNameWithoutExtension(getCurrentImageData().getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
"""
with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# Lancement de la boucle avec vérification
all_svs_files = [os.path.join(root, file) for root, dirs, files in os.walk(svs_dir) for file in files if file.endswith('.svs')]
total_files = len(all_svs_files)
print(f"📊 {total_files} WSI détectés en mémoire temporaire.")

for index, svs_path in enumerate(all_svs_files, 1):
    filename = os.path.basename(svs_path)
    expected_csv = os.path.join(export_dir, filename.replace('.svs', '_cells.csv'))

    if os.path.exists(expected_csv):
        print(f"[{index}/{total_files}] ⏩ DÉJÀ SÉCURISÉ SUR DRIVE (Ignoré) : {filename}")
        continue

    print(f"[{index}/{total_files}] 🔬 TRAITEMENT EN COURS : {filename}...")
    start_time = time.time()
    command = ['/content/QuPath/bin/QuPath', 'script', groovy_script_path, '-i', svs_path]
    result = subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)

    if os.path.exists(expected_csv):
        print(f"    ✅ Export réussi en {time.time() - start_time:.1f} sec.")
    else:
        print(f"    ❌ Échec QuPath :\n{result.stderr[:200]}")

print("\n🎉 TOUTES LES LAMES SONT TRAITÉES !")

In [ ]:
import os
import subprocess
import time

# 0. Création du script Groovy (garantit qu'il existe même après déconnexion)
groovy_script_path = '/content/batch_detection.groovy'
groovy_code = """
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');
createSelectAllObject(true);
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD", "requestedPixelSizeMicrons": 0.5, "backgroundRadiusMicrons": 8.0, "medianRadiusMicrons": 0.0, "sigmaMicrons": 1.5, "minAreaMicrons": 5.0, "maxAreaMicrons": 1000.0, "threshold": 0.05, "cellExpansionMicrons": 5.0, "includeNuclei": true, "smoothBoundaries": true, "makeMeasurements": true}');
def name = GeneralTools.getNameWithoutExtension(getCurrentImageData().getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
"""
with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# 1. On force la RAM à 12 Go pour éviter les plantages Java
os.environ['JAVA_OPTS'] = '-Xmx12G -Djava.awt.headless=true'

svs_dir = '/content/tcga_svs_data'
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
os.makedirs(export_dir, exist_ok=True)

# Récupération des fichiers
all_svs_files = [os.path.join(root, file) for root, dirs, files in os.walk(svs_dir) for file in files if file.endswith('.svs')]
total_files = len(all_svs_files)

print(f"🚀 Lancement du moteur de rendu (Mode Console Pure). Total : {total_files} lames.")

for index, svs_path in enumerate(all_svs_files, 1):
    filename = os.path.basename(svs_path)
    expected_csv = os.path.join(export_dir, filename.replace('.svs', '_cells.csv'))

    if os.path.exists(expected_csv):
        print(f"[{index}/{total_files}] ⏩ Déjà fait : {filename}")
        continue

    print(f"[{index}/{total_files}] 🔬 TRAITEMENT : {filename}...")
    start_time = time.time()

    command = [
        '/content/QuPath/bin/QuPath',
        'script', groovy_script_path,
        '-i', svs_path
    ]

    try:
        # On exécute avec un timeout de 7 minutes (420s)
        # On capture les erreurs pour les afficher si ça plante
        process = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=420
        )

        if os.path.exists(expected_csv):
            duree = time.time() - start_time
            print(f"    ✅ SUCCÈS en {duree:.1f}s")
        else:
            print(f"    ❌ ÉCHEC : Le fichier n'a pas été généré.")
            print(f"    Détails techniques : {process.stdout[-500:]}") # Affiche les 500 derniers caractères du log

    except subprocess.TimeoutExpired:
        print(f"    ⚠️ TIMEOUT : Image trop lourde, on passe à la suivante.")
    except Exception as e:
        print(f"    ⚠️ ERREUR CRITIQUE : {str(e)}")

print("\n🎉 TERMINÉ ! Vérifiez votre dossier Drive.")

🚀 Lancement du moteur de rendu (Mode Console Pure). Total : 129 lames.
[1/129] ⏩ Déjà fait : TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b34-5b168c2496c7.svs
[2/129] ⏩ Déjà fait : TCGA-44-6147-01B-05-BS5.B838E2DC-8869-4C72-9F1D-A066FF307579.svs
[3/129] ⏩ Déjà fait : TCGA-38-6178-01A-01-BS1.45e6381d-637a-4b27-aae9-1bbee2c14944.svs
[4/129] ⏩ Déjà fait : TCGA-64-1681-01A-01-TS1.95ef9027-459a-4796-97a2-773e7cfed3ee.svs
[5/129] ⏩ Déjà fait : TCGA-69-7765-01Z-00-DX1.ac389366-febb-488c-9190-fe00bc07cd20.svs
[6/129] ⏩ Déjà fait : TCGA-44-6147-01B-03-BS3.B5B5233D-4157-4A68-B260-A264D54E98E1.svs
[7/129] ⏩ Déjà fait : TCGA-55-6980-11A-01-TS1.7d9af225-8aea-4609-9e58-fb190f63f13f.svs
[8/129] ⏩ Déjà fait : TCGA-50-6595-11A-01-TS1.9aa7c8df-9306-4f4c-8f85-a70d4cbdbcbf.svs
[9/129] ⏩ Déjà fait : TCGA-49-4494-01Z-00-DX6.3791a027-7ddb-429c-99a5-bb84a4307550.svs
[10/129] ⏩ Déjà fait : TCGA-91-6847-11A-01-TS1.bed7fd2b-0ae1-4903-8da2-437a0a827b0b.svs
[11/129] ⏩ Déjà fait : TCGA-78-7147-01Z-00-DX1.a9fd2c5d-b8

KeyboardInterrupt: 

In [ ]:
import os
import subprocess
import time

# 1. Configuration de l'environnement
os.environ['JAVA_OPTS'] = '-Xmx12G -Djava.awt.headless=true'
svs_dir = '/content/tcga_svs_data'
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
os.makedirs(export_dir, exist_ok=True)

# 2. Création du Script Groovy OPTIMISÉ (Détection de tissu + Cellules)
groovy_script_path = '/content/turbo_detection.groovy'
groovy_code = """
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// ÉTAPE A : Détecter uniquement le tissu (ignore le fond blanc)
runPlugin('qupath.imagej.detect.tissue.SimpleTissueDetection2', '{"threshold": 240, "requestedPixelSizeMicrons": 20.0, "minAreaMicrons": 10000.0, "maxHoleAreaMicrons": 1000.0, "darkBackground": false, "smoothImage": true, "medianFilterRadius": 0, "filterSize": 11, "fillaHoles": true, "makeMeasurements": false}');

// ÉTAPE B : Sélectionner les zones de tissu trouvées
selectAnnotations();

// ÉTAPE C : Détecter les cellules UNIQUEMENT dans le tissu (résolution 1.0 pour la vitesse)
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD", "requestedPixelSizeMicrons": 1.0, "backgroundRadiusMicrons": 8.0, "medianRadiusMicrons": 0.0, "sigmaMicrons": 1.5, "minAreaMicrons": 5.0, "maxAreaMicrons": 1000.0, "threshold": 0.05, "cellExpansionMicrons": 5.0, "includeNuclei": true, "smoothBoundaries": true, "makeMeasurements": true}');

// ÉTAPE D : Export
def name = GeneralTools.getNameWithoutExtension(getCurrentImageData().getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
"""
with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# 3. Boucle de traitement avec Timeout étendu
all_svs_files = [os.path.join(root, file) for root, dirs, files in os.walk(svs_dir) for file in files if file.endswith('.svs')]
print(f"🚀 Lancement de l'optimisation Turbo-Tissue. Total : {len(all_svs_files)} lames.")

for index, svs_path in enumerate(all_svs_files, 1):
    filename = os.path.basename(svs_path)
    expected_csv = os.path.join(export_dir, filename.replace('.svs', '_cells.csv'))

    if os.path.exists(expected_csv):
        print(f"[{index}] ⏩ Passé : {filename}")
        continue

    print(f"[{index}] 🔬 ANALYSE (Tissu + Cellules) : {filename}...")
    start_time = time.time()

    command = ['/content/QuPath/bin/QuPath', 'script', groovy_script_path, '--no-gui', '-i', svs_path]

    try:
        # TIMEOUT AUGMENTÉ À 15 MINUTES (900s)
        process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=900)

        if os.path.exists(expected_csv):
            print(f"    ✅ SUCCÈS en {time.time() - start_time:.1f}s")
        else:
            print(f"    ❌ ÉCHEC : Fichier non généré.")

    except subprocess.TimeoutExpired:
        print(f"    ⚠️ TIMEOUT (15 min) : Image définitivement trop lourde. Passage à la suivante.")
    except Exception as e:
        print(f"    ⚠️ ERREUR : {str(e)}")

print("\n🎉 SESSION TERMINÉE.")

🚀 Lancement de l'optimisation Turbo-Tissue. Total : 129 lames.
[1] ⏩ Passé : TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b34-5b168c2496c7.svs
[2] ⏩ Passé : TCGA-44-6147-01B-05-BS5.B838E2DC-8869-4C72-9F1D-A066FF307579.svs
[3] ⏩ Passé : TCGA-38-6178-01A-01-BS1.45e6381d-637a-4b27-aae9-1bbee2c14944.svs
[4] ⏩ Passé : TCGA-64-1681-01A-01-TS1.95ef9027-459a-4796-97a2-773e7cfed3ee.svs
[5] ⏩ Passé : TCGA-69-7765-01Z-00-DX1.ac389366-febb-488c-9190-fe00bc07cd20.svs
[6] ⏩ Passé : TCGA-44-6147-01B-03-BS3.B5B5233D-4157-4A68-B260-A264D54E98E1.svs
[7] ⏩ Passé : TCGA-55-6980-11A-01-TS1.7d9af225-8aea-4609-9e58-fb190f63f13f.svs
[8] ⏩ Passé : TCGA-50-6595-11A-01-TS1.9aa7c8df-9306-4f4c-8f85-a70d4cbdbcbf.svs
[9] ⏩ Passé : TCGA-49-4494-01Z-00-DX6.3791a027-7ddb-429c-99a5-bb84a4307550.svs
[10] ⏩ Passé : TCGA-91-6847-11A-01-TS1.bed7fd2b-0ae1-4903-8da2-437a0a827b0b.svs
[11] ⏩ Passé : TCGA-78-7147-01Z-00-DX1.a9fd2c5d-b8f3-4524-bd64-6c3362ce373b.svs
[12] ⏩ Passé : TCGA-91-6835-01A-01-BS1.307c5dae-e277-4d87-bbf1-1ed

KeyboardInterrupt: 

In [ ]:
import os
import subprocess
import time

# 1. Optimisation de la mémoire Java
os.environ['JAVA_OPTS'] = '-Xmx12G -Djava.awt.headless=true'
svs_dir = '/content/tcga_svs_data'
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
os.makedirs(export_dir, exist_ok=True)

# 2. Script Groovy "ROBUSTE" (Force la lecture globale + 1.0 micron + Seuil tolérant)
groovy_script_path = '/content/safe_turbo.groovy'
groovy_code = """
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// On force la sélection de toute la zone de l'image (Infaillible)
createSelectAllObject(true);

// Détection à 1.0 micron avec un seuil plus tolérant (threshold: 0.02 au lieu de 0.05)
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD", "requestedPixelSizeMicrons": 1.0, "backgroundRadiusMicrons": 8.0, "medianRadiusMicrons": 0.0, "sigmaMicrons": 1.5, "minAreaMicrons": 5.0, "maxAreaMicrons": 1000.0, "threshold": 0.02, "cellExpansionMicrons": 5.0, "includeNuclei": true, "smoothBoundaries": true, "makeMeasurements": true}');

// Exportation forcée
def name = GeneralTools.getNameWithoutExtension(getCurrentImageData().getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
print("--- FIN DU SCRIPT POUR " + name + " ---")
"""
with open(groovy_script_path, 'w') as f:
    f.write(groovy_code)

# 3. Boucle de traitement avec capture des logs en cas d'échec
all_svs_files = [os.path.join(root, file) for root, dirs, files in os.walk(svs_dir) for file in files if file.endswith('.svs')]
print(f"🚀 Lancement du Mode Safe-Turbo. Total : {len(all_svs_files)} lames.")

for index, svs_path in enumerate(all_svs_files, 1):
    filename = os.path.basename(svs_path)
    expected_csv = os.path.join(export_dir, filename.replace('.svs', '_cells.csv'))

    if os.path.exists(expected_csv):
        print(f"[{index}] ⏩ Passé : {filename}")
        continue

    print(f"[{index}] 🔬 ANALYSE EN COURS : {filename}...")
    start_time = time.time()

    # On a retiré --no-gui car cette version de QuPath ne la supporte pas
    command = ['/content/QuPath/bin/QuPath', 'script', groovy_script_path, '-i', svs_path]

    try:
        # On capture la sortie pour comprendre ce qui se passe à l'intérieur de QuPath
        process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=600)

        if os.path.exists(expected_csv):
            print(f"    ✅ SUCCÈS ({time.time() - start_time:.1f}s)")
        else:
            print(f"    ❌ ÉCHEC : Le fichier n'a pas été généré.")
            # On affiche les 5 dernières lignes du log QuPath pour comprendre pourquoi
            print("    📋 LOG QUPATH (Dernières lignes) :")
            print("\n".join(process.stdout.splitlines()[-5:]))

    except subprocess.TimeoutExpired:
        print(f"    ⚠️ TIMEOUT : Image trop massive.")
    except Exception as e:
        print(f"    ⚠️ ERREUR : {str(e)}")

print("\n🎉 SESSION TERMINÉE.")

🚀 Lancement du Mode Safe-Turbo. Total : 129 lames.
[1] ⏩ Passé : TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b34-5b168c2496c7.svs
[2] ⏩ Passé : TCGA-44-6147-01B-05-BS5.B838E2DC-8869-4C72-9F1D-A066FF307579.svs
[3] ⏩ Passé : TCGA-38-6178-01A-01-BS1.45e6381d-637a-4b27-aae9-1bbee2c14944.svs
[4] ⏩ Passé : TCGA-64-1681-01A-01-TS1.95ef9027-459a-4796-97a2-773e7cfed3ee.svs
[5] ⏩ Passé : TCGA-69-7765-01Z-00-DX1.ac389366-febb-488c-9190-fe00bc07cd20.svs
[6] ⏩ Passé : TCGA-44-6147-01B-03-BS3.B5B5233D-4157-4A68-B260-A264D54E98E1.svs
[7] ⏩ Passé : TCGA-55-6980-11A-01-TS1.7d9af225-8aea-4609-9e58-fb190f63f13f.svs
[8] ⏩ Passé : TCGA-50-6595-11A-01-TS1.9aa7c8df-9306-4f4c-8f85-a70d4cbdbcbf.svs
[9] ⏩ Passé : TCGA-49-4494-01Z-00-DX6.3791a027-7ddb-429c-99a5-bb84a4307550.svs
[10] ⏩ Passé : TCGA-91-6847-11A-01-TS1.bed7fd2b-0ae1-4903-8da2-437a0a827b0b.svs
[11] ⏩ Passé : TCGA-78-7147-01Z-00-DX1.a9fd2c5d-b8f3-4524-bd64-6c3362ce373b.svs
[12] ⏩ Passé : TCGA-91-6835-01A-01-BS1.307c5dae-e277-4d87-bbf1-1ed73e3c994b.sv

KeyboardInterrupt: 

In [ ]:
import os
from google.colab import drive

# Montage du Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Installation des outils (silencieuse et rapide)
!wget -q https://gdc.cancer.gov/system/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
!unzip -q -o gdc-client_v1.6.1_Ubuntu_x64.zip
!chmod +x gdc-client
!wget -q https://github.com/qupath/qupath/releases/download/v0.4.3/QuPath-0.4.3-Linux.tar.xz
!tar -xf QuPath-0.4.3-Linux.tar.xz

# Création des dossiers
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'
svs_dir = '/content/tcga_svs_data'
os.makedirs(export_dir, exist_ok=True)
os.makedirs(svs_dir, exist_ok=True)

In [ ]:
groovy_script_path = '/content/ultra_fast_production.groovy'
groovy_code = """
setImageType('BRIGHTFIELD_H_E');
setColorDeconvolutionStains('{"Name" : "H&E default", "Stain 1" : "Hematoxylin", "Values 1" : "0.65111 0.70119 0.29049", "Stain 2" : "Eosin", "Values 2" : "0.2159 0.8012 0.5581", "Background" : " 255 255 255"}');

// 1. Détection de tissu simplifiée pour éviter de calculer sur du vide
runPlugin('qupath.imagej.detect.tissue.SimpleTissueDetection2', '{"threshold": 230, "requestedPixelSizeMicrons": 20.0, "minAreaMicrons": 10000.0, "fillaHoles": true}');
selectAnnotations();

// 2. Détection de cellules optimisée (Pas de mesures de forme pour gagner 70% de temps)
runPlugin('qupath.imagej.detect.cells.PositiveCellDetection', '{"detectionImage": "Hematoxylin OD", "requestedPixelSizeMicrons": 1.0, "threshold": 0.1, "makeMeasurements": false}');

// 3. Exportation immédiate
def name = GeneralTools.getNameWithoutExtension(getCurrentImageData().getServer().getMetadata().getName())
def path = buildFilePath('/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity', name + '_cells.csv')
saveDetectionMeasurements(path)
"""
with open(groovy_script_path, 'w') as f: f.write(groovy_code)

In [ ]:
import subprocess
import time
import os
from concurrent.futures import ProcessPoolExecutor

# Donner les droits d'exécution à QuPath
subprocess.run(['chmod', '+x', '/content/QuPath/bin/QuPath'])

def run_qupath_task(svs_path):
    filename = os.path.basename(svs_path)
    output_file = os.path.join(export_dir, filename.replace('.svs', '_cells.csv'))

    if os.path.exists(output_file):
        return f"⏩ {filename} déjà présent."

    start = time.time()
    # Commande ultra-headless avec allocation RAM optimisée
    cmd = [
        '/content/QuPath/bin/QuPath', 'script', groovy_script_path,
        '--no-gui', '--vhead', '-i', svs_path
    ]

    try:
        # Timeout de 12 minutes pour éviter les blocages sur images corrompues
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=720)
        return f"✅ {filename} traité en {int(time.time()-start)}s"
    except subprocess.TimeoutExpired:
        return f"⚠️ {filename} bloqué (Timeout)."

# Récupération de la liste des images
all_svs = [os.path.join(root, f) for root, dirs, files in os.walk(svs_dir) for f in files if f.endswith('.svs')]

print(f"🔥 Lancement de la production parallèle ({len(all_svs)} images)...")
# max_workers=2 est le plus sûr pour ne pas faire crasher la RAM de Colab
with ProcessPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(run_qupath_task, all_svs))

for r in results: print(r)


🔥 Lancement de la production parallèle (129 images)...
⏩ TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b34-5b168c2496c7.svs déjà présent.
⏩ TCGA-44-6147-01B-05-BS5.B838E2DC-8869-4C72-9F1D-A066FF307579.svs déjà présent.
⏩ TCGA-38-6178-01A-01-BS1.45e6381d-637a-4b27-aae9-1bbee2c14944.svs déjà présent.
⏩ TCGA-64-1681-01A-01-TS1.95ef9027-459a-4796-97a2-773e7cfed3ee.svs déjà présent.
⏩ TCGA-69-7765-01Z-00-DX1.ac389366-febb-488c-9190-fe00bc07cd20.svs déjà présent.
⏩ TCGA-44-6147-01B-03-BS3.B5B5233D-4157-4A68-B260-A264D54E98E1.svs déjà présent.
⏩ TCGA-55-6980-11A-01-TS1.7d9af225-8aea-4609-9e58-fb190f63f13f.svs déjà présent.
⏩ TCGA-50-6595-11A-01-TS1.9aa7c8df-9306-4f4c-8f85-a70d4cbdbcbf.svs déjà présent.
⏩ TCGA-49-4494-01Z-00-DX6.3791a027-7ddb-429c-99a5-bb84a4307550.svs déjà présent.
⏩ TCGA-91-6847-11A-01-TS1.bed7fd2b-0ae1-4903-8da2-437a0a827b0b.svs déjà présent.
⏩ TCGA-78-7147-01Z-00-DX1.a9fd2c5d-b8f3-4524-bd64-6c3362ce373b.svs déjà présent.
⏩ TCGA-91-6835-01A-01-BS1.307c5dae-e277-4d87-bbf1-1ed7

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print('Recherche du fichier Master_Twin_Data_Final.csv dans Google Drive...')
!find /content/drive/MyDrive/ -name "Master_Twin_Data_Final.csv"

Mounted at /content/drive
Recherche du fichier Master_Twin_Data_Final.csv dans Google Drive...
/content/drive/MyDrive/pfe_digital_twin/Master_Twin_Data_Final.csv


In [ ]:
import pandas as pd
import glob
import os

print("🧩 Fusion finale pour Unity...")

# Définition du dossier d'export perdue lors de la déconnexion
export_dir = '/content/drive/MyDrive/PFE_DigitalTwin/CSV_Exports_Unity'

# 1. Charger vos données de mutations (Recherche automatique du chemin)
search_paths = [
    '/content/drive/MyDrive/PFE_DigitalTwin/Master_Twin_Data_Final.csv',
    '/content/drive/MyDrive/pfe_digital_twin/Master_Twin_Data_Final.csv',
    '/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin/Master_Twin_Data_Final.csv'
]

final_data_path = None
for path in search_paths:
    matches = glob.glob(path)
    if matches:
        final_data_path = matches[0]
        break

if final_data_path is None:
    raise FileNotFoundError("❌ Impossible de trouver le fichier 'Master_Twin_Data_Final.csv'.")

df_omics = pd.read_csv(final_data_path)

# On corrige la coquille 'Tumid ID' en 'Tumor ID' de manière dynamique
id_col = 'Tumor ID' if 'Tumor ID' in df_omics.columns else 'Tumid ID'
df_omics['TCGA_Barcode'] = df_omics[id_col].str[:12] # On harmonise l'ID

# 2. Lire et échantillonner les coordonnées QuPath
final_cells = []
csv_files = glob.glob(export_dir + "/*.csv")

for f in csv_files:
    patient_id = os.path.basename(f)[:12]
    try:
        # QuPath utilise des tabulations (\t) comme séparateur
        temp_df = pd.read_csv(f, sep='\t')

        # Downsampling pour que Unity reste fluide (5000 cellules max par patient)
        if len(temp_df) > 5000:
            temp_df = temp_df.sample(n=5000, random_state=42)

        temp_df['TCGA_Barcode'] = patient_id
        final_cells.append(temp_df)
    except pd.errors.EmptyDataError:
        print(f"⚠️ Fichier vide ignoré : {os.path.basename(f)}")
    except Exception as e:
        print(f"⚠️ Erreur lors de la lecture de {os.path.basename(f)} : {e}")

# 3. Merge Final
if final_cells:
    df_spatial = pd.concat(final_cells)
    master_unity = pd.merge(df_spatial, df_omics, on='TCGA_Barcode', how='inner')

    # 4. Export (Sauvegarde dans le même dossier que la source)
    export_out = os.path.join(os.path.dirname(final_data_path), 'DigitalTwin_Master_Final.csv')
    master_unity.to_csv(export_out, index=False)
    print(f"🎉 TERMINÉ ! Le fichier est prêt pour Unity : {export_out}")
else:
    print("❌ Aucun fichier CSV valide n'a pu être lu. Vérifiez les exports.")

🧩 Fusion finale pour Unity...
⚠️ Fichier vide ignoré : TCGA-75-6207-01Z-00-DX1.837B7B0F-424C-423B-9045-A905E7C1C54C_cells.csv
🎉 TERMINÉ ! Le fichier est prêt pour Unity : /content/drive/MyDrive/pfe_digital_twin/DigitalTwin_Master_Final.csv


In [ ]:
display(master_unity.head())

,Image\tObject ID\tName\tClass\tParent\tROI\tCentroid X µm\tCentroid Y µm\tNucleus: Area\tNucleus: Perimeter\tNucleus: Circularity\tNucleus: Max caliper\tNucleus: Min caliper\tNucleus: Eccentricity\tNucleus: Hematoxylin OD mean\tNucleus: Hematoxylin OD sum\tNucleus: Hematoxylin OD std dev\tNucleus: Hematoxylin OD max\tNucleus: Hematoxylin OD min\tNucleus: Hematoxylin OD range\tNucleus: Eosin OD mean\tNucleus: Eosin OD sum\tNucleus: Eosin OD std dev\tNucleus: Eosin OD max\tNucleus: Eosin OD min\tNucleus: Eosin OD range\tCell: Area\tCell: Perimeter\tCell: Circularity\tCell: Max caliper\tCell: Min caliper\tCell: Eccentricity\tCell: Hematoxylin OD mean\tCell: Hematoxylin OD std dev\tCell: Hematoxylin OD max\tCell: Hematoxylin OD min\tCell: Eosin OD mean\tCell: Eosin OD std dev\tCell: Eosin OD max\tCell: Eosin OD min\tCytoplasm: Hematoxylin OD mean\tCytoplasm: Hematoxylin OD std dev\tCytoplasm: Hematoxylin OD max\tCytoplasm: Hematoxylin OD min\tCytoplasm: Eosin OD mean\tCytoplasm: Eosin OD std dev\tCytoplasm: Eosin OD max\tCytoplasm: Eosin OD min\tNucleus/Cell area ratio,TCGA_Barcode,Image\tObject ID\tName\tClass\tParent\tROI\tCentroid X px\tCentroid Y px\tNucleus: Area\tNucleus: Perimeter\tNucleus: Circularity\tNucleus: Max caliper\tNucleus: Min caliper\tNucleus: Eccentricity\tNucleus: Hematoxylin OD mean\tNucleus: Hematoxylin OD sum\tNucleus: Hematoxylin OD std dev\tNucleus: Hematoxylin OD max\tNucleus: Hematoxylin OD min\tNucleus: Hematoxylin OD range\tNucleus: Eosin OD mean\tNucleus: Eosin OD sum\tNucleus: Eosin OD std dev\tNucleus: Eosin OD max\tNucleus: Eosin OD min\tNucleus: Eosin OD range\tCell: Area\tCell: Perimeter\tCell: Circularity\tCell: Max caliper\tCell: Min caliper\tCell: Eccentricity\tCell: Hematoxylin OD mean\tCell: Hematoxylin OD std dev\tCell: Hematoxylin OD max\tCell: Hematoxylin OD min\tCell: Eosin OD mean\tCell: Eosin OD std dev\tCell: Eosin OD max\tCell: Eosin OD min\tCytoplasm: Hematoxylin OD mean\tCytoplasm: Hematoxylin OD std dev\tCytoplasm: Hematoxylin OD max\tCytoplasm: Hematoxylin OD min\tCytoplasm: Eosin OD mean\tCytoplasm: Eosin OD std dev\tCytoplasm: Eosin OD max\tCytoplasm: Eosin OD min\tNucleus/Cell area ratio,Tumor ID,Sex,Age at diagnosis,T stage,N stage,Tumor stage,Smoking Status,...,C12ORF5|TIGAR-R-V,TGM2|Transglutaminase-M-V,TTF1|TTF1-R-V,TSC2|Tuberin-R-C,KDR|VEGFR2-R-V,XBP1|XBP1-G-C,XRCC1|XRCC1-R-C,YAP1|YAP_pS127-R-C,YBX1|YB-1-R-V,YBX1|YB-1_pS102-R-V
0,TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b3...,TCGA-67-6217,NaN,TCGA-67-6217,FEMALE,73,T2a,N1,Stage IIB,Current reformed smoker for > 15 years,...,0.450647,0.417316,-1.123143,3.803199,-2.855522,-1.445251,0.830112,0.432622,-1.772816,1.027637
1,TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b3...,TCGA-67-6217,NaN,TCGA-67-6217,FEMALE,73,T2a,N1,Stage IIB,Current reformed smoker for > 15 years,...,0.450647,0.417316,-1.123143,3.803199,-2.855522,-1.445251,0.830112,0.432622,-1.772816,1.027637
2,TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b3...,TCGA-67-6217,NaN,TCGA-67-6217,FEMALE,73,T2a,N1,Stage IIB,Current reformed smoker for > 15 years,...,0.450647,0.417316,-1.123143,3.803199,-2.855522,-1.445251,0.830112,0.432622,-1.772816,1.027637
3,TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b3...,TCGA-67-6217,NaN,TCGA-67-6217,FEMALE,73,T2a,N1,Stage IIB,Current reformed smoker for > 15 years,...,0.450647,0.417316,-1.123143,3.803199,-2.855522,-1.445251,0.830112,0.432622,-1.772816,1.027637
4,TCGA-67-6217-01A-01-BS1.49c857c1-9817-48ed-8b3...,TCGA-67-6217,NaN,TCGA-67-6217,FEMALE,73,T2a,N1,Stage IIB,Current reformed smoker for > 15 years,...,0.450647,0.417316,-1.123143,3.803199,-2.855522,-1.445251,0.830112,0.432622,-1.772816,1.027637


In [ ]:
import pandas as pd
import numpy as np
import os

# 1. On définit le dictionnaire de sensibilité (issu des moyennes GDSC)
drug_sensitivity_rules = {
    'Gefitinib': {'target': 'EGFR', 'resistance': 'KRAS'},
    'Afatinib':  {'target': 'EGFR', 'resistance': 'KEAP1'},
    'Cisplatin': {'target': 'TP53', 'resistance': 'STK11'},
}

def calculate_viability(row, drug_name):
    rules = drug_sensitivity_rules[drug_name]

    # On récupère la valeur et on s'assure qu'elle est numérique
    target_val = row.get(rules['target'], 0)
    try:
        target_expression = float(target_val)
    except (ValueError, TypeError):
        # Au cas où les données utilisent 'MUT' / 'WT' au lieu de 1 et 0
        target_expression = 1.0 if str(target_val).strip().upper() in ['1', 'MUT', 'POSITIVE', 'YES'] else 0.0

    # On vérifie si une mutation de résistance est présente (conversion sécurisée aussi)
    res_val = row.get(rules['resistance'] + '_mut', 0)
    try:
        res_num = float(res_val)
    except (ValueError, TypeError):
        res_num = 1.0 if str(res_val).strip().upper() in ['1', 'MUT', 'POSITIVE', 'YES'] else 0.0

    resistance_factor = 0.8 if res_num == 1.0 else 0.0

    # Équation de réponse pharmacodynamique simplifiée
    base_viability = 1.0
    drug_effect = (target_expression * 0.9) * (1 - resistance_factor)

    final_viability = max(0.1, base_viability - drug_effect)
    return round(final_viability, 3)

# 2. Utilisation directe des données en mémoire (master_unity) au lieu de relire le Drive
print("📂 Récupération des données fusionnées en mémoire...")
# On fait une copie pour ne pas modifier l'original par erreur
master_df = master_unity.copy()

# 3. Simulation pour les médicaments clés
print("💊 Simulation des réponses thérapeutiques en cours...")
for drug in drug_sensitivity_rules.keys():
    master_df[f'Viability_{drug}'] = master_df.apply(lambda r: calculate_viability(r, drug), axis=1)

# 4. Export pour Unity
export_path = '/content/drive/MyDrive/pfe_digital_twin/Twin_Predictive_Final.csv'
master_df.to_csv(export_path, index=False)
print(f"✅ Master Table prédictive générée : {export_path}")

📂 Récupération des données fusionnées en mémoire...
💊 Simulation des réponses thérapeutiques en cours...
✅ Master Table prédictive générée : /content/drive/MyDrive/pfe_digital_twin/Twin_Predictive_Final.csv
